# 01 — Core ICNN Lyapunov Experiments: full Exp5 → Exp7 pipeline

This notebook is intentionally **self-contained**. It does **not** require an existing Exp5 checkpoint.

It reproduces the full procedure:

1. generate/load random derivative data for `damped_pendulum_4`,
2. train the Exp5-style phase-1 stable ICNN model on derivative MSE,
3. train the baseline MLP on derivative MSE,
4. take the newly trained phase-1 `fhat`,
5. generate rollout states using that `fhat`,
6. build `random states + rollout states`,
7. freeze `fhat`,
8. train only the ICNN Lyapunov network `V`,
9. compare `baseline`, `phase1 stable`, `fhat only`, and `Exp7-style stable`.

The key point is that the Exp7-style stage is run **after** the notebook has trained its own Exp5-style nominal dynamics.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "stable_icnn_physics").exists():
            return p
    raise RuntimeError("Start Jupyter from the DeepLearningFTN repository root.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from stable_icnn_physics import BaselineDynamicsMLP, build_stable_model, make_system
from stable_icnn_physics.data import (
    dataset_base_name,
    generate_derivative_data,
    load_dataset,
    save_dataset,
    tensor_dataset,
)
from stable_icnn_physics.eval import (
    autoregressive_rollout_model,
    lyapunov_decrease_values,
    rollout_error,
    rollout_system,
)
from stable_icnn_physics.train import evaluate_derivative_mse, train_derivative_model
from stable_icnn_physics.rollout_augmented import (
    collect_rollout_states,
    projection_diagnostics,
    train_lyapunov_v_only_on_states,
)

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CACHE_DIR = REPO_ROOT / "data" / "cache"
CKPT_DIR = REPO_ROOT / "checkpoints"
RESULTS_DIR = REPO_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "core_icnn_plots"

for d in [CACHE_DIR, CKPT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("device:", DEVICE, "| torch:", torch.__version__)

## Configuration

In [ ]:
SEED = 0
TOLERANCE = 1e-5

SYSTEM_NAME = "damped_pendulum_4"
SYSTEM_KWARGS = {"friction": 0.3, "gravity": 9.81}

DT = 0.02
ROLLOUT_STEPS = 300
TRAIN_SAMPLES = 50_000
TEST_SAMPLES = 10_000
EVAL_TRAJS = 16

# Architecture/hyperparameters matched to the successful Exp5/Exp7 runs.
HIDDEN = 200
DEPTH = 3
LYAPUNOV_HIDDEN = 100
LYAPUNOV_EPS = 0.01
ALPHA = 1e-5
REHU_WIDTH = 0.01

# Full from-scratch procedure.
PHASE1_EPOCHS = 500
PHASE1_BATCH_SIZE = 256
PHASE1_LR = 1e-3

N_ROLLOUT_TRAIN_TRAJS = 500
N_ROLLOUT_VAL_TRAJS = 100
V_ONLY_EPOCHS = 400
V_ONLY_BATCH_SIZE = 512
V_ONLY_LR = 1e-3
V_ONLY_ETA_MIN = 1e-4

TAG = "core_p4_full_exp5_exp7"

# Keep these True when you want the notebook to reproduce everything.
# Set to False only when you intentionally want to reuse this notebook's own checkpoints.
FORCE_RETRAIN_PHASE1 = True
FORCE_RETRAIN_V_ONLY = True

PHASE1_STABLE_CKPT = CKPT_DIR / f"{TAG}_phase1_exp5_stable.pt"
PHASE1_BASELINE_CKPT = CKPT_DIR / f"{TAG}_phase1_exp5_baseline.pt"
EXP7_STYLE_CKPT = CKPT_DIR / f"{TAG}_exp7style_stable.pt"

print("FORCE_RETRAIN_PHASE1:", FORCE_RETRAIN_PHASE1)
print("FORCE_RETRAIN_V_ONLY:", FORCE_RETRAIN_V_ONLY)

## 1. Generate/load derivative data

In [ ]:
system = make_system(SYSTEM_NAME, **SYSTEM_KWARGS)
state_dim = system.state_dim

for split, n in [("train", TRAIN_SAMPLES), ("test", TEST_SAMPLES)]:
    path = CACHE_DIR / dataset_base_name(
        system, split=split, n_samples=n, seed=SEED, dataset_type="derivative"
    )
    if not path.exists():
        print(f"generating {split}: {path.name}")
        x, y = generate_derivative_data(system, n_samples=n, split=split, seed=SEED)
        save_dataset(path, x, y, metadata={"system": system.metadata(), "split": split})
    else:
        print(f"reusing {split}: {path.name}")

train_path = CACHE_DIR / dataset_base_name(
    system, split="train", n_samples=TRAIN_SAMPLES, seed=SEED, dataset_type="derivative"
)
test_path = CACHE_DIR / dataset_base_name(
    system, split="test", n_samples=TEST_SAMPLES, seed=SEED, dataset_type="derivative"
)

x_train, y_train = load_dataset(train_path)
x_test, y_test = load_dataset(test_path)
train_ds = tensor_dataset(x_train, y_train)
test_ds = tensor_dataset(x_test, y_test)

x0_eval = system.sample_initial_conditions(EVAL_TRAJS, split="test", seed=SEED + 123)
true_traj = rollout_system(system, x0_eval, steps=ROLLOUT_STEPS, dt=DT)

print("system:", system.name, "| state_dim:", state_dim)
print("train:", x_train.shape, y_train.shape)
print("test:", x_test.shape, y_test.shape)
print("eval x0:", x0_eval.shape)

## 2. Model builders and helpers

In [ ]:
def make_stable_model():
    return build_stable_model(
        dim=state_dim,
        hidden=HIDDEN,
        depth=DEPTH,
        lyapunov_hidden=LYAPUNOV_HIDDEN,
        lyapunov_eps=LYAPUNOV_EPS,
        alpha=ALPHA,
        rehu_width=REHU_WIDTH,
    )

def make_baseline_model():
    return BaselineDynamicsMLP(dim=state_dim, hidden=HIDDEN, depth=DEPTH)

def summarize_error(err: np.ndarray) -> dict[str, float]:
    return {"final": float(err[-1]), "mean": float(err.mean()), "max": float(err.max())}

def rollout_and_error(model, name: str):
    traj = autoregressive_rollout_model(
        model,
        x0_eval,
        steps=ROLLOUT_STEPS,
        dt=DT,
        device=DEVICE,
        wrap_fn=system.wrap_state,
    )
    err = rollout_error(system, true_traj, traj).mean(axis=1)
    print(f"{name:>22s} | final={err[-1]:.6g} mean={err.mean():.6g} max={err.max():.6g}")
    return traj, err

## 3. Phase 1 — train Exp5-style models inside the notebook

This is the part that was missing in the stale notebook.

The notebook now trains:

- `phase1_stable`: ICNN-projected stable model on derivative MSE,
- `baseline`: unconstrained MLP on derivative MSE.

The `fhat` used later comes from this newly trained `phase1_stable`, not from an external checkpoint.

In [ ]:
if FORCE_RETRAIN_PHASE1 or not PHASE1_STABLE_CKPT.exists() or not PHASE1_BASELINE_CKPT.exists():
    phase1_stable = make_stable_model()
    baseline = make_baseline_model()

    print("Phase 1 stable params:", sum(p.numel() for p in phase1_stable.parameters()))
    print("Baseline params:", sum(p.numel() for p in baseline.parameters()))

    print("\n[Phase 1] training stable ICNN-projected model on derivative MSE...")
    hist_phase1_stable = train_derivative_model(
        phase1_stable,
        train_ds,
        test_ds,
        epochs=PHASE1_EPOCHS,
        batch_size=PHASE1_BATCH_SIZE,
        learning_rate=PHASE1_LR,
        device=DEVICE,
        checkpoint_path=PHASE1_STABLE_CKPT,
        print_every=max(1, PHASE1_EPOCHS // 10),
        use_amp=False,
    )

    print("\n[Phase 1] training baseline MLP on derivative MSE...")
    hist_baseline = train_derivative_model(
        baseline,
        train_ds,
        test_ds,
        epochs=PHASE1_EPOCHS,
        batch_size=PHASE1_BATCH_SIZE,
        learning_rate=PHASE1_LR,
        device=DEVICE,
        checkpoint_path=PHASE1_BASELINE_CKPT,
        print_every=max(1, PHASE1_EPOCHS // 10),
        use_amp=(DEVICE == "cuda"),
    )
else:
    print("Loading this notebook's phase-1 checkpoints")
    phase1_stable = make_stable_model()
    phase1_stable.load_state_dict(
        torch.load(PHASE1_STABLE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
    )
    baseline = make_baseline_model()
    baseline.load_state_dict(
        torch.load(PHASE1_BASELINE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
    )
    hist_phase1_stable = None
    hist_baseline = None

phase1_stable.to(DEVICE).eval()
baseline.to(DEVICE).eval()

print("phase1 stable ckpt:", PHASE1_STABLE_CKPT)
print("baseline ckpt:", PHASE1_BASELINE_CKPT)

## 4. Phase 1 evaluation before rollout-augmented V training

In [ ]:
dmse_phase1_stable = evaluate_derivative_mse(phase1_stable, test_ds, device=DEVICE)
dmse_fhat = evaluate_derivative_mse(phase1_stable.fhat, test_ds, device=DEVICE)
dmse_baseline = evaluate_derivative_mse(baseline, test_ds, device=DEVICE)

print("Derivative MSE:")
print("  phase1 stable projected:", dmse_phase1_stable)
print("  fhat only:              ", dmse_fhat)
print("  baseline:               ", dmse_baseline)

phase1_stable_traj, err_phase1_stable = rollout_and_error(phase1_stable, "phase1 stable")
fhat_traj, err_fhat = rollout_and_error(phase1_stable.fhat, "fhat only")
baseline_traj, err_baseline = rollout_and_error(baseline, "baseline")

## 5. Generate rollout states using the newly trained `fhat`

In [ ]:
print("Collecting rollout states using phase1_stable.fhat...")

rollout_train_states = collect_rollout_states(
    phase1_stable.fhat,
    system,
    n_trajs=N_ROLLOUT_TRAIN_TRAJS,
    steps=ROLLOUT_STEPS,
    dt=DT,
    split="train",
    seed=SEED + 1,
    device=DEVICE,
)
rollout_val_states = collect_rollout_states(
    phase1_stable.fhat,
    system,
    n_trajs=N_ROLLOUT_VAL_TRAJS,
    steps=ROLLOUT_STEPS,
    dt=DT,
    split="test",
    seed=SEED + 2,
    device=DEVICE,
)

combined_train_states = np.concatenate([x_train, rollout_train_states], axis=0)
combined_val_states = np.concatenate([x_test, rollout_val_states], axis=0)

print("rollout train states:", rollout_train_states.shape)
print("rollout val states:", rollout_val_states.shape)
print("combined train states:", combined_train_states.shape)
print("combined val states:", combined_val_states.shape)

## 6. Phase 2 — Exp7-style V-only training

This stage starts from the phase-1 stable model, freezes `fhat`, and trains only the ICNN Lyapunov function `V` on random + rollout states.

In [ ]:
exp7_stable = make_stable_model()
exp7_stable.load_state_dict(
    torch.load(PHASE1_STABLE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
)
exp7_stable.to(DEVICE).eval()

if FORCE_RETRAIN_V_ONLY or not EXP7_STYLE_CKPT.exists():
    history_v = train_lyapunov_v_only_on_states(
        exp7_stable,
        train_states=combined_train_states,
        val_states=combined_val_states,
        epochs=V_ONLY_EPOCHS,
        batch_size=V_ONLY_BATCH_SIZE,
        learning_rate=V_ONLY_LR,
        eta_min=V_ONLY_ETA_MIN,
        device=DEVICE,
        print_every=40,
    )
    torch.save({"model_state": exp7_stable.state_dict(), "history": history_v.__dict__}, EXP7_STYLE_CKPT)
    print("saved:", EXP7_STYLE_CKPT)
else:
    exp7_stable.load_state_dict(
        torch.load(EXP7_STYLE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
    )
    history_v = None
    print("loaded:", EXP7_STYLE_CKPT)

exp7_stable.to(DEVICE).eval()

## 7. V-only training curve

In [ ]:
if history_v is not None:
    plt.figure(figsize=(8, 4))
    plt.semilogy(history_v.epoch, history_v.train_loss, label="train")
    plt.semilogy(history_v.epoch, history_v.val_loss, label="val")
    plt.semilogy(history_v.epoch, history_v.best_val_loss, label="best val", linestyle="--")
    plt.xlabel("epoch")
    plt.ylabel("Lyapunov violation loss")
    plt.title("Exp7-style V-only training")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / f"{TAG}_vonly_training_loss.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)
else:
    print("No new V-only training history; checkpoint was loaded.")

## 8. Final evaluation after Exp7-style V training

In [ ]:
dmse_exp7_stable = evaluate_derivative_mse(exp7_stable, test_ds, device=DEVICE)
dmse_exp7_fhat = evaluate_derivative_mse(exp7_stable.fhat, test_ds, device=DEVICE)

print("Derivative MSE after V-only training:")
print("  exp7 projected:", dmse_exp7_stable)
print("  fhat only:     ", dmse_exp7_fhat)
print("  baseline:      ", dmse_baseline)

exp7_stable_traj, err_exp7_stable = rollout_and_error(exp7_stable, "exp7 stable")
exp7_fhat_traj, err_exp7_fhat = rollout_and_error(exp7_stable.fhat, "fhat only")

## 9. Lyapunov/projection diagnostics

In [ ]:
decrease_phase1 = lyapunov_decrease_values(phase1_stable, x_test[:2048], device=DEVICE).ravel()
decrease_exp7 = lyapunov_decrease_values(exp7_stable, x_test[:2048], device=DEVICE).ravel()

diag_phase1 = projection_diagnostics(phase1_stable, phase1_stable_traj, device=DEVICE)
diag_exp7 = projection_diagnostics(exp7_stable, exp7_stable_traj, device=DEVICE)

print("Phase1 projection fire rate:", diag_phase1["projection_fire_rate"])
print("Exp7 projection fire rate:  ", diag_exp7["projection_fire_rate"])
print("Phase1 correction mean:", diag_phase1["correction_norm_mean"])
print("Exp7 correction mean:  ", diag_exp7["correction_norm_mean"])

## 10. Save summary

In [ ]:
summary = {
    "experiment": TAG,
    "system": system.metadata(),
    "dt": DT,
    "rollout_steps": ROLLOUT_STEPS,
    "train_samples": TRAIN_SAMPLES,
    "test_samples": TEST_SAMPLES,
    "phase1": {
        "stable_ckpt": str(PHASE1_STABLE_CKPT.relative_to(REPO_ROOT)),
        "baseline_ckpt": str(PHASE1_BASELINE_CKPT.relative_to(REPO_ROOT)),
        "derivative_mse_stable_projected": float(dmse_phase1_stable),
        "derivative_mse_fhat_only": float(dmse_fhat),
        "derivative_mse_baseline": float(dmse_baseline),
        "rollout_phase1_stable": summarize_error(err_phase1_stable),
        "rollout_fhat_only": summarize_error(err_fhat),
        "rollout_baseline": summarize_error(err_baseline),
        "lyapunov_max_violation_projected": float(decrease_phase1.max()),
        "lyapunov_fraction_satisfied_projected": float(np.mean(decrease_phase1 <= TOLERANCE)),
        "projection_fire_rate": diag_phase1["projection_fire_rate"],
        "correction_norm_mean": diag_phase1["correction_norm_mean"],
        "correction_norm_p95": diag_phase1["correction_norm_p95"],
        "correction_norm_max": diag_phase1["correction_norm_max"],
    },
    "exp7_style": {
        "stable_ckpt": str(EXP7_STYLE_CKPT.relative_to(REPO_ROOT)),
        "derivative_mse_stable_projected": float(dmse_exp7_stable),
        "derivative_mse_fhat_only": float(dmse_exp7_fhat),
        "derivative_mse_baseline": float(dmse_baseline),
        "rollout_exp7_stable": summarize_error(err_exp7_stable),
        "rollout_fhat_only": summarize_error(err_exp7_fhat),
        "rollout_baseline": summarize_error(err_baseline),
        "lyapunov_max_violation_projected": float(decrease_exp7.max()),
        "lyapunov_fraction_satisfied_projected": float(np.mean(decrease_exp7 <= TOLERANCE)),
        "projection_fire_rate": diag_exp7["projection_fire_rate"],
        "correction_norm_mean": diag_exp7["correction_norm_mean"],
        "correction_norm_p95": diag_exp7["correction_norm_p95"],
        "correction_norm_max": diag_exp7["correction_norm_max"],
        "nominal_violation_mean": diag_exp7["nominal_violation_mean"],
        "nominal_violation_max": diag_exp7["nominal_violation_max"],
        "discrete_dV_frac_nonpositive": diag_exp7["discrete_dV_frac_nonpositive"],
        "discrete_dV_max": diag_exp7["discrete_dV_max"],
    },
}

summary_path = RESULTS_DIR / f"{TAG}_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("saved:", summary_path)

## 11. Rollout error plots

In [ ]:
t = np.arange(ROLLOUT_STEPS + 1) * DT

plt.figure(figsize=(9, 5))
plt.plot(t, err_baseline, label="baseline MLP")
plt.plot(t, err_fhat, label="fhat only")
plt.plot(t, err_phase1_stable, label="phase1 stable projected")
plt.plot(t, err_exp7_stable, label="Exp7-style projected")
plt.xlabel("time [s]")
plt.ylabel("mean state error")
plt.title("Pendulum-4 rollout error: Exp5 → Exp7-style")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_rollout_error.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

plt.figure(figsize=(9, 5))
plt.semilogy(t, err_baseline + 1e-12, label="baseline MLP")
plt.semilogy(t, err_fhat + 1e-12, label="fhat only")
plt.semilogy(t, err_phase1_stable + 1e-12, label="phase1 stable projected")
plt.semilogy(t, err_exp7_stable + 1e-12, label="Exp7-style projected")
plt.xlabel("time [s]")
plt.ylabel("mean state error, log scale")
plt.title("Pendulum-4 rollout error log-scale")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_rollout_error_log.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

## 12. Projection diagnostics plots

In [ ]:
t_mid = t[:-1]

plt.figure(figsize=(8, 4))
plt.plot(t_mid, diag_phase1["fire"].mean(axis=1), label="phase1")
plt.plot(t_mid, diag_exp7["fire"].mean(axis=1), label="Exp7-style")
plt.xlabel("time [s]")
plt.ylabel("projection fire rate")
plt.ylim(-0.02, 1.02)
plt.title("Projection fire rate over rollout")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_projection_fire_rate.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

plt.figure(figsize=(8, 4))
plt.plot(t_mid, diag_phase1["correction_norm"].mean(axis=1), label="phase1 mean")
plt.plot(t_mid, diag_exp7["correction_norm"].mean(axis=1), label="Exp7 mean")
plt.plot(t_mid, np.quantile(diag_exp7["correction_norm"], 0.95, axis=1), label="Exp7 p95", linestyle="--")
plt.xlabel("time [s]")
plt.ylabel("|| correction ||")
plt.title("Projection correction magnitude")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_correction_norm.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

plt.figure(figsize=(8, 4))
plt.hist(diag_exp7["violation"].ravel(), bins=80)
plt.axvline(0.0, linestyle="--")
plt.xlabel("gradV · fhat + alpha V")
plt.ylabel("count")
plt.title("Nominal Lyapunov violation after Exp7-style training")
plt.grid(True, alpha=0.3)
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_nominal_violation_hist.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

## 13. Example trajectories

In [ ]:
coord_names = [f"theta_{i}" for i in range(4)] + [f"omega_{i}" for i in range(4)]
traj_id = 0

for j, name in enumerate(coord_names):
    plt.figure(figsize=(9, 3.8))
    plt.plot(t, true_traj[:, traj_id, j], label="true")
    plt.plot(t, baseline_traj[:, traj_id, j], label="baseline")
    plt.plot(t, fhat_traj[:, traj_id, j], label="fhat only")
    plt.plot(t, phase1_stable_traj[:, traj_id, j], label="phase1 stable")
    plt.plot(t, exp7_stable_traj[:, traj_id, j], label="Exp7-style stable")
    plt.xlabel("time [s]")
    plt.ylabel(name)
    plt.title(f"Pendulum-4 coordinate: {name}")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    path = PLOTS_DIR / f"{TAG}_traj_{j}_{name}.png"
    plt.savefig(path, dpi=160)
    plt.show()

## Report interpretation

Use this notebook as the reproducible source for the main claim:

> Joint training of the projected stable model can learn a good nominal vector field `fhat`, but the learned Lyapunov function can induce destructive projection during rollout. A second stage that freezes `fhat` and trains only the ICNN Lyapunov function on random + model-induced rollout states reduces projection bias and improves long-horizon autoregressive behaviour.

The relevant comparison is:

```text
phase1 stable projected vs fhat only vs Exp7-style projected
```